### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
        unique_name="maps_router_eta_1m",
    version_from_unique_name="maps_router_eta",
    version_comment="""
We randomly sub-sample the train to 1 million and test data 250k rows. We follow TabReD and use random sub-sampling. The idea behind this instead of a time-based subsampling is to keep data from various time periods and model the distribution shift across the full time horizon.
""",
    # same as maps_router_eta.ipynb
    dataset_year="2024",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/pcovkrd84mejm/maps-routing",
    download_description="""
We get the TabRed data from Kaggle.

kaggle datasets download -d pcovkrd84mejm/maps-routing -f maps_routing.parquet && unzip maps_routing.parquet.zip && rm maps_routing.parquet.zip && mkdir -p local-data-warehouse/maps_router_eta && mv maps_routing.parquet local-data-warehouse/maps_router_eta/
""",
    # References
    academic_reference_bibtex="""@inproceedings{rubachev2025tabred,
  title={TabReD: Analyzing Pitfalls and Filling the Gaps in Tabular Deep Learning Benchmarks},
  author={Rubachev, Ivan and Kartashev, Nikolay and Gorishniy, Yury and Babenko, Artem},
  booktitle={The Thirteenth International Conference on Learning Representations},
  year={2025},
}
""",
    academic_reference_bibtex_key="rubachev2025tabred",
    license="CC-BY-NC-SA-4.0",
    data_tags=["Non-IID", "Temporal", "Anonymized"],
    curation_comments="""
We start with data from TabRed, which already comes preprocessed.
We perform no additional preprocessing steps.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target_log_spkm",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="timestamp",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_parquet(dataset_mold.path / "maps_routing.parquet")
print("Loaded data shape:", df.shape)

df["timestamp"] = pd.to_datetime(df["timestamp"])

# We take all bin + cat as Category
cat_cols = [c for c in df.columns if c.startswith("cat") or c.startswith("bin")]
df[cat_cols] = df[cat_cols].astype("category")

df = df.sort_values(by="timestamp").reset_index(drop=True)

Loaded data shape: (13639272, 989)


## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import subsample_temporal

date_col = task_mold.time_on
target_col = task_mold.target_column_name

df = df.sort_values(by=date_col).reset_index(drop=True)

test_time_min = df[date_col].max().normalize() - pd.DateOffset(weeks=1)

# Define indices
test_idx = df.index[df[date_col] >= test_time_min].to_numpy().tolist()
train_idx = df.index[
    df[date_col] < test_time_min
].to_numpy().tolist()

df, train_idx, test_idx = subsample_temporal(
    df=df,
    train_idx=train_idx,
    test_idx=test_idx,
    stratify_on=task_mold.stratify_on,
)

print("Train size:", len(train_idx), "| Test size:", len(test_idx))
print("Train target mean:", df.loc[train_idx, target_col].mean())
print("Test target mean:", df.loc[test_idx, target_col].mean())

splits = {0: {0: (train_idx, test_idx)}}

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We use the last week as test data and all prior data as train data.",
    splits=splits,
    time_horizon=7,
    time_horizon_unit="days",
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)